## Consideraciones sobre `merge`

Cuando se unen tablas con `merge` mediante una columna de union, si un valor de esa
columna **no existe en la otra tabla**, esas filas desaparecen de la tabla resultante.

> No es que falte el dato, es que no encuentra pareja en la otra tabla.

### Error silencioso nº 21

`pd.merge` por defecto descarta las filas que no encuentran pareja.
No da error ni aviso, el codigo funciona, simplemente el resultado tiene
menos filas de las que deberia.

**Como detectarlo:** comparar el numero de filas antes y despues del merge.

### Solucion: `how='left'`

Hay que indicar de que tabla queremos que permanezcan todas las filas mediante el
argumento `how`. En este caso `'left'` indica que la tabla de la izquierda.

Las filas sin pareja se conservan, rellenando con `NaN` las columnas que no ha
podido traer. Asi el problema es **visible** en lugar de quedar oculto.

```python
df_seguro = pd.merge(df, islas_incompleta, on='island', how='left')
print(df_seguro.shape)   # (344, 9) -> no se pierde ningun pinguino
```

In [1]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset('penguins')
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [2]:
islas = pd.DataFrame({
    'island': ['Biscoe', 'Dream', 'Torgersen'],
    'superficie_km2': [45.2, 12.8, 0.4],
    'base_permanente': [True, False, True]
})
islas

,island,superficie_km2,base_permanente
0,Biscoe,45.2,True
1,Dream,12.8,False
2,Torgersen,0.4,True


In [3]:
df_completo = pd.merge(df, islas, on='island')
print(df_completo.shape)
df_completo.head()


(344, 9)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,superficie_km2,base_permanente
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,0.4,True
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,0.4,True
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,0.4,True
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,0.4,True
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,0.4,True


In [4]:
islas_incompleta = islas[islas['island'] != 'Dream']
print(islas_incompleta)

df_roto = pd.merge(df, islas_incompleta, on='island')
print(df_roto.shape)
print(df_roto)

      island  superficie_km2  base_permanente
0     Biscoe            45.2             True
2  Torgersen             0.4             True
(220, 9)
    species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0    Adelie  Torgersen            39.1           18.7              181.0   
1    Adelie  Torgersen            39.5           17.4              186.0   
2    Adelie  Torgersen            40.3           18.0              195.0   
3    Adelie  Torgersen             NaN            NaN                NaN   
4    Adelie  Torgersen            36.7           19.3              193.0   
..      ...        ...             ...            ...                ...   
215  Gentoo     Biscoe             NaN            NaN                NaN   
216  Gentoo     Biscoe            46.8           14.3              215.0   
217  Gentoo     Biscoe            50.4           15.7              222.0   
218  Gentoo     Biscoe            45.2           14.8              212.0   
219  Gentoo     B

In [5]:
df_seguro = pd.merge(df, islas_incompleta, on='island', how='left')
print(df_seguro.shape)

(344, 9)


In [6]:
df_seguro[df_seguro['island'] == 'Dream'].head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,superficie_km2,base_permanente
30,Adelie,Dream,39.5,16.7,178.0,3250.0,Female,NaN,NaN
31,Adelie,Dream,37.2,18.1,178.0,3900.0,Male,NaN,NaN
32,Adelie,Dream,39.5,17.8,188.0,3300.0,Female,NaN,NaN
33,Adelie,Dream,40.9,18.9,184.0,3900.0,Male,NaN,NaN
34,Adelie,Dream,36.4,17.0,195.0,3325.0,Female,NaN,NaN


## Tipos de union en `merge` — parametro `how`

La regla: **`how` nombra la tabla cuyas filas se conservan todas.**

| `how` | Que conserva | Riesgo |
|---|---|---|
| `'inner'` (por defecto) | Solo lo que aparece en **ambas** tablas | Pierde filas en silencio |
| `'left'` | Todas las de la tabla **izquierda** | Genera NaN, pero visibles |
| `'right'` | Todas las de la tabla **derecha** | Pierde filas de la izquierda |
| `'outer'` | Todas las de **las dos** tablas | Genera NaN por ambos lados |

En la practica `'left'` es el mas usado: normalmente hay una tabla principal
(la que quiero conservar entera) y otra de la que solo quiero traer informacion.

`'outer'` es util para **auditar**: muestra las filas huerfanas de los dos lados,
que es justo lo que quiero detectar antes de decidir como unir.

In [7]:
# right: se queda con todas las filas de la tabla de la DERECHA
df_right = pd.merge(df, islas_incompleta, on='island', how='right')
print('right :', df_right.shape)

# outer: se queda con TODAS las filas de las dos tablas
df_outer = pd.merge(df, islas_incompleta, on='island', how='outer')
print('outer :', df_outer.shape)

right : (220, 9)
outer : (344, 9)


In [8]:
islas_fantasma = pd.concat([
    islas_incompleta,
    pd.DataFrame({'island': ['Antarctica'], 'superficie_km2': [99.9], 'base_permanente': [True]})
])
print(islas_fantasma)

print('left  :', pd.merge(df, islas_fantasma, on='island', how='left').shape)
print('outer :', pd.merge(df, islas_fantasma, on='island', how='outer').shape)

       island  superficie_km2  base_permanente
0      Biscoe            45.2             True
2   Torgersen             0.4             True
0  Antarctica            99.9             True
left  : (344, 9)
outer : (345, 9)


In [9]:
res = pd.merge(df, islas_fantasma, on='island', how='outer')
res[res['species'].isna()]

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,superficie_km2,base_permanente
0,NaN,Antarctica,NaN,NaN,NaN,NaN,NaN,99.9,True


## Error silencioso nº 22: `merge` que multiplica filas

Cuando `merge` **pierde** filas al menos tenemos una pista: la tabla resultante
es mas pequeña de lo esperado.

Lo peligroso es cuando la tabla de referencia **ya tenia duplicados** en la
columna de union. El merge no los crea, los saca a la luz: cada fila de la
izquierda que encuentra dos parejas se duplica en el resultado.

### Por que es tan peligroso

La tabla **crece** en lugar de encoger, y una tabla que crece no levanta sospechas.
A partir de ahi ninguna agregacion es fiable:

```python
df['body_mass_g'].mean()               # 4201.75 -> el valor real
df_multiplicado['body_mass_g'].mean()  # ~4500   -> falso, pero creible
```

La suma se dispara y canta. La media solo se **desplaza**: no da un resultado
absurdo, da un resultado plausible pero falso. Un resultado absurdo se detecta,
uno plausible se publica.

### Como detectarlo antes de cruzar

```python
# ¿Cuantos duplicados hay en la columna de union?
islas_duplicada['island'].duplicated().sum()

# ¿Cuales son? (para decidir a mano con cual quedarse)
islas_duplicada[islas_duplicada['island'].duplicated(keep=False)]
```

### Como prevenirlo: `validate`

Con `validate` declaramos **que relacion esperamos entre las dos tablas**.
Si la realidad no cumple lo declarado, pandas lanza un `MergeError` y para.

```python
pd.merge(df, islas_duplicada, on='island', how='left', validate='many_to_one')
# MergeError: Merge keys are not unique in right dataset; not a many-to-one merge
```

Se lee siempre **izquierda_a_derecha**, segun el orden en que escribimos
las tablas. Repetido = `many`, unico = `one`.

| Valor | Significado |
|---|---|
| `'many_to_one'` | Izquierda repite, derecha unica. **El mas comun** |
| `'one_to_many'` | Izquierda unica, derecha repite |
| `'one_to_one'` | Ninguna de las dos repite. El mas estricto |
| `'many_to_many'` | Repiten las dos. No valida nada y multiplica |

Truco: el `many` va del lado de la tabla grande.

### Conclusion

Un error que **salta** es un regalo. Lo que arruina un analisis no es el codigo
que falla, es el codigo que funciona haciendo algo distinto de lo que creiamos.

In [10]:
islas_duplicada = pd.DataFrame({
    'island': ['Biscoe', 'Biscoe', 'Dream', 'Torgersen'],
    'superficie_km2': [45.2, 46.0, 12.8, 0.4],
    'base_permanente': [True, True, False, True]
})
islas_duplicada

,island,superficie_km2,base_permanente
0,Biscoe,45.2,True
1,Biscoe,46.0,True
2,Dream,12.8,False
3,Torgersen,0.4,True


In [11]:
df_multiplicado = pd.merge(df, islas_duplicada, on='island', how='left')
print(df_multiplicado.shape)

(512, 9)


In [12]:
df_multiplicado[df_multiplicado['island'] == 'Biscoe'].head(4)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,superficie_km2,base_permanente
20,Adelie,Biscoe,37.8,18.3,174.0,3400.0,Female,45.2,True
21,Adelie,Biscoe,37.8,18.3,174.0,3400.0,Female,46.0,True
22,Adelie,Biscoe,37.7,18.7,180.0,3600.0,Male,45.2,True
23,Adelie,Biscoe,37.7,18.7,180.0,3600.0,Male,46.0,True


In [13]:
print('Original     :', df['body_mass_g'].mean())
print('Multiplicado :', df_multiplicado['body_mass_g'].mean())

Original     : 4201.754385964912
Multiplicado : 4370.481335952849


In [14]:
print(islas_duplicada['island'].duplicated().sum())
print(islas_duplicada[islas_duplicada['island'].duplicated(keep=False)])

1
   island  superficie_km2  base_permanente
0  Biscoe            45.2             True
1  Biscoe            46.0             True


In [15]:
df_validado = pd.merge(df, islas_duplicada, on='island', how='left', validate='many_to_one')

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

## `concat` — apilar tablas

`merge` y `concat` resuelven problemas distintos:

- **`merge`** cruza **a lo ancho**: mismas filas, mas columnas. **Enriquece**.
- **`concat`** apila **a lo alto**: mismas columnas, mas filas. **Acumula**.

El caso tipico de `concat`: los datos estan partidos en trozos con la misma
estructura (ventas de enero, febrero, marzo...) y queremos una sola tabla.

Recibe una **lista** de dataframes, no dos argumentos sueltos:

```python
juntos = pd.concat([primeros, ultimos])
```

Por eso se pueden apilar 12 meses de golpe pasandole una lista de 12.

---

### Error silencioso nº 23: indices duplicados

`concat` **no reinicia el indice**, se trae el de cada trozo tal cual.
Al apilar, las etiquetas se repiten.

```python
solapadas = pd.concat([df.head(100), df.head(50)])
solapadas.index.duplicated().sum()   # 50 indices repetidos
```

**Por que importa:** `.loc` busca por **etiqueta**. Si la etiqueta 5 existe
dos veces, `.loc[5]` deja de devolver una fila y devuelve un DataFrame de dos.
Todo el codigo escrito asumiendo "una etiqueta = una fila" se rompe:
a veces peta, y a veces (lo peor) funciona y devuelve un resultado equivocado.

**Solucion:**

```python
juntos_ok = pd.concat([df.head(100), df.head(50)], ignore_index=True)
juntos_ok.index                        # RangeIndex(start=0, stop=150, step=1)
juntos_ok.index.duplicated().sum()     # 0
```

`RangeIndex` es la señal de que el indice esta limpio: una cuenta 0,1,2,3...

**Cuidado — dos problemas distintos:**
`ignore_index=True` arregla el **indice duplicado**, NO elimina **filas duplicadas**.
La tabla sigue teniendo 150 filas y las 50 primeras siguen estando dos veces.

**Regla practica:** al apilar, `ignore_index=True` casi siempre.
Solo se omite si el indice significa algo real que queremos conservar (ej: una fecha).

---

### Error silencioso nº 24: columnas que no coinciden

`concat` empareja las columnas **por nombre, literalmente**. No entiende que
`peso` y `body_mass_g` son lo mismo: para pandas son dos columnas sin relacion.
Un espacio de mas, una mayuscula o un acento ya las convierte en distintas.

Y **no da error**: crea columnas nuevas y rellena todo lo que falta con `NaN`.

```python
otra = pd.DataFrame({'especie': ['Adelie'], 'peso': [3750]})
mezcla = pd.concat([df.head(3), otra], ignore_index=True)
mezcla.shape        # (4, 9)  <- 7 columnas originales + 2 nuevas
```

**Por que es grave:** el peso 3750 queda en la columna `peso`, no en
`body_mass_g`. Al calcular `mezcla['body_mass_g'].mean()` ese pinguino
**no cuenta**, aunque su fila este ahi delante.

Caso real: apilamos 12 ficheros mensuales y en marzo alguien renombro una
columna. Once meses van bien, marzo se va a una columna nueva, y el total
anual se come un mes entero sin ningun aviso.

**Como comprobarlo antes de apilar:**

```python
df.columns.equals(otra.columns)      # False -> no coinciden

set(df.columns) - set(otra.columns)  # esta en df pero no en otra
set(otra.columns) - set(df.columns)  # esta en otra pero no en df
```

`set()` convierte los nombres en un **conjunto** (sin orden, sin repetidos).
Entre conjuntos, el `-` significa "lo que esta en el primero y no en el segundo".
El orden importa: `a - b` y `b - a` dan resultados distintos, por eso se
escriben las dos lineas.

**Solucion:** renombrar antes de apilar.

```python
otra = otra.rename(columns={'peso': 'body_mass_g', 'especie': 'species'})
unidas = pd.concat([df.head(3), otra], ignore_index=True)
unidas.shape        # (4, 7)  <- ahora si
```

---

### NaN honestos vs NaN por fallo de estructura

Tras el `rename`, la ultima fila sigue teniendo `NaN` en cinco columnas.
Pero es un `NaN` **honesto**: ese pinguino realmente no tiene medida de pico
ni de aleta. Su peso esta donde debe estar y **cuenta** en las agregaciones.

Antes del `rename` los `NaN` reflejaban un fallo mio de estructura.
En pantalla se ven igual. Solo yo se cual es cual.

In [16]:
primeros = df.head(100)
ultimos = df.tail(50)

print(primeros.shape)
print(ultimos.shape)

juntos = pd.concat([primeros, ultimos])
print(juntos.shape)

(100, 7)
(50, 7)
(150, 7)


In [17]:
juntos.index

Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
       334, 335, 336, 337, 338, 339, 340, 341, 342, 343],
      dtype='int64', length=150)

In [19]:
solapadas = pd.concat([df.head(100), df.head(50)])
print(solapadas.shape)
print(solapadas.index.duplicated().sum())
print(solapadas.loc[5])

(150, 7)
50
  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
5  Adelie  Torgersen            39.3           20.6              190.0   
5  Adelie  Torgersen            39.3           20.6              190.0   

   body_mass_g   sex  
5       3650.0  Male  
5       3650.0  Male  


In [20]:
juntos_ok = pd.concat([df.head(100), df.head(50)], ignore_index=True)
print(juntos_ok.index[:5])
print(juntos_ok.index.duplicated().sum())

RangeIndex(start=0, stop=5, step=1)
0


In [21]:
otra = pd.DataFrame({'especie': ['Adelie'], 'peso': [3750]})
mezcla = pd.concat([df.head(3), otra], ignore_index=True)
print(mezcla.shape)
mezcla

(4, 9)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,especie,peso
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,NaN,NaN
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,NaN,NaN
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adelie,3750.0


In [22]:
print(df.columns.equals(otra.columns))

# Y si no coinciden, ver exactamente en qué se diferencian:
print(set(df.columns) - set(otra.columns))
print(set(otra.columns) - set(df.columns))

False
{'island', 'bill_depth_mm', 'body_mass_g', 'sex', 'bill_length_mm', 'species', 'flipper_length_mm'}
{'especie', 'peso'}


In [25]:
otra = otra.rename(columns={'peso': 'body_mass_g', 'especie': 'species'})

unidas = pd.concat([df.head(3),otra], ignore_index= True)
unidas

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,NaN,NaN,NaN,NaN,3750.0,NaN
